In [66]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
import mujoco_playground as mp
from mujoco_playground import registry 

from gpe.utils import acting
from gpe.utils.models import MHPolicy
from brax.envs.wrappers.training import VmapWrapper, EpisodeWrapper
from gpe.utils.buffer import TrajectorySamplingQueue
from gpe.pref_pe import prefill_buffer
from gpe.utils import types

import jax
import jax.numpy as jnp

import flax.nnx as nnx

In [41]:
# registry.ALL_ENVS

seed = 0
key = jax.random.PRNGKey(seed)
rngs = nnx.Rngs(seed)

In [14]:
task = 'CartpoleBalance'
default_env = registry.load(task)
env = acting.wrap_env_for_training(default_env, 200)

In [16]:
env_key = jax.random.split(key, 1)
env_state = env.reset(env_key)

In [17]:
env_state.obs.shape

(1, 5)

In [18]:
a = jnp.zeros(shape=(1, env.action_size))
env_state = env.step(env_state, a)

In [19]:
obs_dim, act_dim = env.observation_size, env.action_size
policy_model = MHPolicy(
    rngs=rngs,
    obs_dim=obs_dim,
    act_dim=act_dim,
    beta=1.0,
    hidden_size=256,
)

In [20]:
obs = env_state.obs
init_action = env_state.info["init_action"]
act, _ = policy_model(obs, init_action, rngs())

In [22]:
_, transitions = acting.generate_unroll(env, env_state, policy_model, rngs(), 5)

In [90]:
dummy_obs = jnp.zeros((1, obs_dim))
dummy_action = jnp.zeros((1, act_dim))
dummy_zero = jnp.zeros((1,))
dummy_transition = types.Transition(  # pytype: disable=wrong-arg-types  # jax-ndarray
    observation=dummy_obs,
    action=dummy_action,
    reward=dummy_zero,
    discount=dummy_zero,
    next_observation=dummy_obs,
    extras={"state_extras": {"truncation": dummy_zero}},
)

buffer = TrajectorySamplingQueue(
    max_replay_size=206,
    dummy_data_sample=dummy_transition,
    sample_batch_size=10,
    horizon=1,
)
key, buffer_key = jax.random.split(key)
buffer_state = buffer.init(buffer_key)

In [91]:
env_state, buffer_state = prefill_buffer(
    key=key,
    env=env,
    env_state=env_state,
    buffer_state=buffer_state,
    policy=policy_model,
    buffer=buffer,
    num_itr=203
)

In [92]:
buffer_state, batch = buffer.sample(buffer_state)

In [93]:
buffer_state.mask[:1000]

Array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0.,
       0., 0.], dtype=float32)